In [0]:
-- Main databases are created, fraud_raw for all datasets, and fraud_unified as a conjunction of all.

CREATE DATABASE IF NOT EXISTS fraud_raw;
CREATE DATABASE IF NOT EXISTS fraud_unified;


-- Now we upload existing datasets into the data lake fraud_raw

CREATE TABLE fraud_raw.paysim AS
SELECT *
FROM workspace.final_project_data.synthetic_financial_datasets_for_fraud_detection;

CREATE TABLE fraud_raw.fraud_detection_2023 AS
SELECT *
FROM workspace.final_project_data.final_transactions;

CREATE TABLE fraud_raw.fraud_detection_2025 AS
SELECT *
FROM workspace.final_project_data.financial_fraud_detection_dataset;

CREATE TABLE fraud_raw.baf_base AS
SELECT *
FROM workspace.final_project_data.base;

CREATE TABLE fraud_raw.baf_variant_3 AS
SELECT *
FROM workspace.final_project_data.variant_iii;

CREATE TABLE fraud_raw.baf_variant_5 AS
SELECT *
FROM workspace.final_project_data.variant_v;


-- Here I am creting bronze schema

CREATE DATABASE IF NOT EXISTS fraud_bronze;

CREATE DATABASE IF NOT EXISTS fraud_silver;

SHOW TABLES IN fraud_bronze;

-- I am creating gold database

CREATE DATABASE IF NOT EXISTS fraud_gold;

-- Drop country column based on ds validation

ALTER TABLE fraud_gold.fraud_2023_standarized SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);

ALTER TABLE fraud_gold.fraud_2023_standarized
DROP COLUMN country;


ALTER TABLE fraud_gold.paysim_standarized SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);

ALTER TABLE fraud_gold.paysim_standarized
DROP COLUMN country;

ALTER TABLE fraud_gold.fraud_2023_standarized SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);
ALTER TABLE fraud_gold.fraud_2023_standarized 
ALTER COLUMN channel TYPE STRING;

-- 1. Add a new temporary string column
ALTER TABLE fraud_gold.fraud_2023_standarized ADD COLUMN channel_str STRING;

-- 2. Copy the data over, casting it to string in the process
UPDATE fraud_gold.fraud_2023_standarized SET channel_str = CAST(channel AS STRING);

-- 3. Drop the old BIGINT column (instant metadata drop!)
ALTER TABLE fraud_gold.fraud_2023_standarized DROP COLUMN channel;

-- 4. Rename the new column to the original name
ALTER TABLE fraud_gold.fraud_2023_standarized RENAME COLUMN channel_str TO channel;


ALTER TABLE fraud_gold.fraud_2023_standarized SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);

ALTER TABLE fraud_gold.fraud_2023_standarized
DROP COLUMN account_balance;

ALTER TABLE fraud_gold.fraud_2025_standarized SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
);

ALTER TABLE fraud_gold.fraud_2025_standarized
DROP COLUMN account_balance;

SELECT COUNT(is_fraud)
FROM fraud_gold.fraud_2025_standarized
WHERE is_fraud = 0;

DROP TABLE fraud_silver.baf_transactions;

SELECT DISTINCT(dataset_source)
FROM fraud_gold.account_application_fraud